# Notebook 02: Land Use Land Cover (LULC) & Distance to Water Calculation

This notebook covers reprojecting and standardizing the Land Use / Land Cover (LULC) dataset, and calculating the Euclidean distance to waterbodies (critical hydrology layer for HSI).

## Objectives:
1. Reproject and clip Dynamic World LULC layer.
2. Extract waterbodies (Class 0 in Dynamic World).
3. Compute Euclidean Distance to waterbodies in meters using scipy EDT.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

# Configure paths
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import clean_raster, reproject_raster
from src.distance import calculate_distance_to_water

print('Environment ready!')

## 1. Reproject and Clip Dynamic World LULC

We load the raw 2024 Dynamic World GeoTIFF, reproject it using **nearest-neighbor** resampling (since land cover categories are discrete integers), and clip it to the park boundary.

In [ ]:
raw_lulc = PROJECT_ROOT / 'data' / 'raw' / 'DynamicWorld' / 'DynamicWorld_2024.tif'
clipped_red = PROJECT_ROOT / 'data' / 'processed' / 'Cleaned' / 'B04.tif'
aoi_path = PROJECT_ROOT / 'data' / 'raw' / 'AOI' / 'Corbett_AOI.shp'
aoi = gpd.read_file(aoi_path)

dw_reprojected = PROJECT_ROOT / 'data' / 'processed' / 'Cleaned' / 'DynamicWorld_Reprojected.tif'
dw_clean = PROJECT_ROOT / 'data' / 'processed' / 'Cleaned' / 'DynamicWorld_Clean.tif'

# Reproject LULC (discrete=True)
reproject_raster(raw_lulc, clipped_red, dw_reprojected, is_discrete=True)

# Clip LULC to AOI
clean_raster(dw_reprojected, dw_clean, aoi, nodata_value=255)

if dw_reprojected.exists():
    os.remove(dw_reprojected)

# Visualize LULC classes
with rasterio.open(dw_clean) as src:
    lulc_arr = src.read(1)
    
plt.figure(figsize=(8,8))
plt.imshow(lulc_arr, cmap='tab10', vmin=0, vmax=9)
plt.colorbar(label='LULC Category Code')
plt.title('Processed LULC (Dynamic World)')
plt.axis('off')
plt.show()

## 2. Calculate Distance to Waterbody

Wildlife requires close proximity to water sources. Class 0 in Dynamic World represents water bodies. We extract this mask, calculate the distance in pixels to the closest water cell, and multiply by pixel resolution (10m) to get the distance in meters.

In [ ]:
distance_clean = PROJECT_ROOT / 'data' / 'processed' / 'Cleaned' / 'DistanceToWater_Clean.tif'
calculate_distance_to_water(dw_clean, distance_clean, resolution=10.0)

# Visualize Distance to Water
with rasterio.open(distance_clean) as src:
    dist_arr = src.read(1)
    
plt.figure(figsize=(8,8))
plt.imshow(dist_arr, cmap='viridis')
plt.colorbar(label='Distance (meters)')
plt.title('Distance to Waterbody')
plt.axis('off')
plt.show()